In [ ]:
import sys; sys.path.insert(0, "../src")
import pandas as pd, numpy as np
from elo import add_elo
from features import add_form

df = pd.read_csv("../data/processed/laliga_matches.csv",
                 parse_dates=["Date"], dtype={"Season": str})
df = add_elo(df)
df = add_form(df)

# Partido de control: Betis, últimos partidos
b = df[(df["HomeTeam"] == "Betis") | (df["AwayTeam"] == "Betis")].head(8)
print(b[["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "h_GF_f", "a_GF_f"]])

         Date   HomeTeam     AwayTeam  FTHG  FTAG    h_GF_f  a_GF_f
2  2016-08-20  Barcelona        Betis     6     2       NaN     NaN
10 2016-08-26      Betis    La Coruna     0     0  2.000000    2.00
29 2016-09-11   Valencia        Betis     2     3  1.000000    1.00
30 2016-09-16      Betis      Granada     2     2  1.666667    1.00
41 2016-09-20    Sevilla        Betis     1     0  2.250000    1.75
50 2016-09-23      Betis       Malaga     1     0  1.400000    1.00
60 2016-09-30   Sociedad        Betis     1     0  1.600000    1.20
73 2016-10-15      Betis  Real Madrid     1     6  1.200000    2.20


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import log_loss

df["y"] = df["FTR"].map({"H": 0, "D": 1, "A": 2})

feats_forma = ["d_puntos", "d_GF", "d_GC", "d_SF", "d_SC", "d_STF", "d_STC", "d_descanso"]

train = df[df["Season"].isin(["1617","1718","1819","1920","2021","2122","2223"])]
test  = df[df["Season"].isin(["2324","2425","2526"])]

def evaluar(cols):
    tr = train.dropna(subset=cols); te = test.dropna(subset=cols)
    mod = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    mod.fit(tr[cols], tr["y"])
    P = mod.predict_proba(te[cols])
    return round(log_loss(te["y"], P, labels=[0, 1, 2]), 4), len(te)

print("Elo solo:       ", evaluar(["elo_diff"]))
print("Forma sola:     ", evaluar(feats_forma))
print("Elo + forma:    ", evaluar(["elo_diff"] + feats_forma))

Elo solo:        (0.9746, 1140)
Forma sola:      (1.0061, 1139)
Elo + forma:     (0.9782, 1139)


In [4]:
esperadas_prefijos = ("h_", "a_", "d_", "elo_")
print([c for c in df.columns if not c.startswith(esperadas_prefijos)])
print(len(df.columns), df.columns.is_unique)

['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR', 'PSH', 'PSD', 'PSA', 'PSCH', 'PSCD', 'PSCA', 'AvgH', 'AvgD', 'AvgA', 'AvgCH', 'AvgCD', 'AvgCA', 'B365CH', 'B365CD', 'B365CA', 'Season', 'y']
62 True


In [5]:
# Mismos partidos para todos: los que tienen todas las variables
todas = ["elo_diff"] + feats_forma
te_comun = test.dropna(subset=todas)
tr_comun = train.dropna(subset=todas)
print("Partidos comunes en test:", len(te_comun))

def evaluar_comun(cols):
    mod = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    mod.fit(tr_comun[cols], tr_comun["y"])
    P = mod.predict_proba(te_comun[cols])
    return round(log_loss(te_comun["y"], P, labels=[0, 1, 2]), 4)

candidatos = {
    "Elo solo":              ["elo_diff"],
    "Elo + d_puntos":        ["elo_diff", "d_puntos"],
    "Elo + d_GF + d_GC":     ["elo_diff", "d_GF", "d_GC"],
    "Elo + d_STF + d_STC":   ["elo_diff", "d_STF", "d_STC"],
    "Elo + d_descanso":      ["elo_diff", "d_descanso"],
    "Elo + todas":           todas,
}
for nombre, cols in candidatos.items():
    print(f"{nombre:22s} {evaluar_comun(cols)}")

Partidos comunes en test: 1139
Elo solo               0.975
Elo + d_puntos         0.9757
Elo + d_GF + d_GC      0.9757
Elo + d_STF + d_STC    0.9744
Elo + d_descanso       0.976
Elo + todas            0.9782


## Feature engineering: forma, tiros y descanso

Variables calculadas con `shift(1)` (sin fuga de datos), ventana de 5 partidos,
como diferencia local − visitante. Verificado a mano con los primeros partidos
del Betis.

| Modelo (1.139 partidos, test 2023/24-2025/26) | Log-loss |
|---|---|
| Elo solo | 0,9750 |
| Elo + puntos | 0,9757 |
| Elo + goles | 0,9757 |
| Elo + tiros a puerta | 0,9744 |
| Elo + descanso | 0,9760 |
| Elo + todas | 0,9782 |

**Lectura:** ninguna variante se distingue del Elo solo (diferencias de
milésimas, muy por debajo del ruido del test). El Elo ya resume la información
de la forma reciente, y añadir todas las variables empeora algo (posible
sobreajuste).